In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, TensorDataset
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import plotly.express as px
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

# Optuna Visualization Tools
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_slice
from optuna.visualization import plot_param_importances

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ==============================
# 1. LOAD & CONFIG
# ==============================
CHOSEN_CROP = "rice" 
TARGET_COL = f"Y_{CHOSEN_CROP}"
PARQUET_PATH = '../../Parquet/XY_v3.parquet'
SEQ_LEN     = 5              # try larger windows; model uses past SEQ_LEN years
BATCH_SIZE  = 32
CLIP_QUANT  = 0.01           # 1%-99% clipping to tame outliers
EPOCHS      = 100
PATIENCE    = 15
LR          = 2e-4

df = pd.read_parquet(PARQUET_PATH)
df = df[(df["year"] >= 1982) & (df["year"] <= 2023)].copy()
df = df.dropna(subset=[TARGET_COL]).copy()
print(f"--> Filtered years (1982-2023) and dropped NaN targets. New Shape: {df.shape}")

LAG_1_FEATURE = f'avg_yield_{CHOSEN_CROP}_1y'


In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

# ============================================================
# 1. PREPROCESSING + SEQUENCE BUILDING  (WITH VERBOSE PRINTS)
# ============================================================
def preprocess_and_build_loaders(
    df: pd.DataFrame,
    chosen_crop: str,
    target_col: str,
    seq_len: int = 5,
    batch_size: int = 32,
    clip_quant: float = 0.01,
):
    """
    Full preprocessing + sequence-building pipeline for panel time series LSTM.
    Now with verbose prints so you can see what each step is doing.
    """

    print("========== PREPROCESSING START ==========")
    print(f"Chosen crop: {chosen_crop}")
    print(f"Target col:  {target_col}")
    print(f"Seq len:     {seq_len}")
    print(f"Batch size:  {batch_size}")
    print("-----------------------------------------")

    df_model = df.copy()
    print(f"Initial df shape: {df_model.shape}")

    # -------------------------------------------
    # 1) Drop rows with NaN target
    # -------------------------------------------
    n_before = len(df_model)
    df_model = df_model[~df_model[target_col].isna()].copy()
    n_after = len(df_model)
    print(f"[Step 1] Dropped rows with NaN target: {n_before - n_after} rows removed")
    print(f"        Remaining rows: {n_after}")

    # -------------------------------------------
    # 1.5) Remove min/max target per country (area)
    # -------------------------------------------
    print("[Step 1.5] Removing per-area min/max target rows as outliers...")

    # ensure we still have area + target
    if "area" not in df_model.columns:
        raise ValueError("'area' column not found in df for min/max removal")

    # indices of min and max target per area
    idx_min = df_model.groupby("area")[target_col].idxmin().values
    idx_max = df_model.groupby("area")[target_col].idxmax().values

    rows_to_drop = np.unique(np.concatenate([idx_min, idx_max]))
    n_before_mm = len(df_model)
    df_model = df_model.loc[~df_model.index.isin(rows_to_drop)].reset_index(drop=True)
    n_after_mm = len(df_model)

    print(f"        Dropped {n_before_mm - n_after_mm} rows (min + max per area).")
    print(f"        Remaining rows after outlier removal: {n_after_mm}")

    # -------------------------------------------
    # 2) Target & feature selection
    # -------------------------------------------
    print("[Step 2] Target & feature selection...")
    # Drop other target columns
    all_targets = [c for c in df_model.columns if c.startswith("Y_")]
    drop_other_targets = [c for c in all_targets if c != target_col]
    df_model = df_model.drop(columns=drop_other_targets)
    print(f"        Target columns found: {all_targets}")
    print(f"        Keeping target: {target_col}")
    print(f"        Dropping other targets: {drop_other_targets}")

    # Drop unrelated avg_yield_* columns
    avg_yield_cols = [c for c in df_model.columns if c.startswith("avg_yield_")]
    chosen_prefix = f"avg_yield_{chosen_crop}_"
    keep_avg_yield = [c for c in avg_yield_cols if c.startswith(chosen_prefix)]
    drop_avg_yield = list(set(avg_yield_cols) - set(keep_avg_yield))
    df_model = df_model.drop(columns=drop_avg_yield)

    print(f"        Total avg_yield_* cols: {len(avg_yield_cols)}")
    print(f"        Keeping {len(keep_avg_yield)} for {chosen_crop}")
    print(f"        Dropping {len(drop_avg_yield)} from other crops")

    # -------------------------------------------
    # 3) Add time features (trend over years)
    # -------------------------------------------
    print("[Step 3] Adding time features year_index and year_norm...")
    base_year = df_model["year"].min()
    df_model["year_index"] = df_model["year"] - base_year
    max_index = df_model["year_index"].max() if df_model["year_index"].max() > 0 else 1
    df_model["year_norm"] = df_model["year_index"] / max_index
    print(f"        Base year: {base_year}")
    print(f"        Max year_index: {max_index}")
    print(f"        year range in df: {df_model['year'].min()}–{df_model['year'].max()}")

    # -------------------------------------------
    # 4) Sort for time consistency
    # -------------------------------------------
    print("[Step 4] Sorting by ['area', 'year']...")
    df_model = df_model.sort_values(["area", "year"]).reset_index(drop=True)
    print(f"        After sort shape: {df_model.shape}")

    # -------------------------------------------
    # 5) Define feature columns
    # -------------------------------------------
    feature_cols = [
        c
        for c in df_model.columns
        if c not in ["area", "year", target_col] and not c.startswith("Y_")
    ]
    print("[Step 5] Defining feature columns...")
    print(f"        Selected {len(feature_cols)} input features for prediction.")
    print(f"        First 10 features: {feature_cols[:10]}")

    # -------------------------------------------
    # 6) Group-wise ffill/bfill for feature columns
    # -------------------------------------------
    print("[Step 6] Group-wise ffill/bfill within each area...")
    na_before = df_model[feature_cols].isna().sum().sum()
    df_model[feature_cols] = (
        df_model.groupby("area", group_keys=False)[feature_cols]
        .apply(lambda g: g.ffill().bfill())
    )
    na_after = df_model[feature_cols].isna().sum().sum()
    print(f"        NaNs before ffill/bfill: {na_before}")
    print(f"        NaNs after  ffill/bfill: {na_after}")

    # -------------------------------------------
    # 7) Time-based masks (on rows; for scaling only)
    # -------------------------------------------
    print("[Step 7] Creating time-based masks (rows)...")
    train_mask_rows = df_model["year"] < 2014
    val_mask_rows   = (df_model["year"] >= 2014) & (df_model["year"] <= 2018)
    test_mask_rows  = df_model["year"] >= 2019

    print(f"        Train rows: {int(train_mask_rows.sum())}")
    print(f"        Val rows:   {int(val_mask_rows.sum())}")
    print(f"        Test rows:  {int(test_mask_rows.sum())}")

    # -------------------------------------------
    # 8) Raw arrays + NaN imputation (train-mean)
    # -------------------------------------------
    print("[Step 8] Building raw arrays and imputing remaining NaNs with train mean...")
    X_raw = df_model[feature_cols].values.astype(np.float32)
    y_raw = df_model[[target_col]].values.astype(np.float32)

    train_X = X_raw[train_mask_rows]
    train_mean = np.nanmean(train_X, axis=0)
    nan_inds = np.where(np.isnan(X_raw))
    n_nan_total = len(nan_inds[0])
    print(f"        Remaining NaNs before mean-impute: {n_nan_total}")
    X_raw[nan_inds] = np.take(train_mean, nan_inds[1])
    print("        NaNs after mean-impute: 0")

    # -------------------------------------------
    # 9) Outlier clipping (train-based quantiles)
    # -------------------------------------------
    if clip_quant is not None and 0.0 < clip_quant < 0.5:
        print(f"[Step 9] Clipping outliers using train quantiles {clip_quant:.2f} and {1-clip_quant:.2f}...")
        train_X_no_nan = X_raw[train_mask_rows]
        q_low = np.quantile(train_X_no_nan, clip_quant, axis=0)
        q_high = np.quantile(train_X_no_nan, 1.0 - clip_quant, axis=0)

        n_clipped_cols = 0
        for j in range(train_X_no_nan.shape[1]):
            if q_low[j] < q_high[j]:
                X_raw[:, j] = np.clip(X_raw[:, j], q_low[j], q_high[j])
                n_clipped_cols += 1
        print(f"        Columns clipped: {n_clipped_cols}/{train_X_no_nan.shape[1]}")
    else:
        print("[Step 9] Skipping clipping (clip_quant is None or invalid).")

    # -------------------------------------------
    # 10) Scaling (fit only on train)
    # -------------------------------------------
    print("[Step 10] Scaling X and y (StandardScaler, train-only fit)...")
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_scaled = X_raw.copy()
    y_scaled = y_raw.copy()

    X_scaled[train_mask_rows] = scaler_X.fit_transform(X_raw[train_mask_rows])
    X_scaled[val_mask_rows]   = scaler_X.transform(X_raw[val_mask_rows])
    X_scaled[test_mask_rows]  = scaler_X.transform(X_raw[test_mask_rows])

    y_scaled[train_mask_rows] = scaler_y.fit_transform(y_raw[train_mask_rows])
    y_scaled[val_mask_rows]   = scaler_y.transform(y_raw[val_mask_rows])
    y_scaled[test_mask_rows]  = scaler_y.transform(y_raw[test_mask_rows])

    print("        Scaling complete.")
    print(f"        y_scaled train mean (should be ~0): {y_scaled[train_mask_rows].mean():.4f}")
    print(f"        y_scaled train std  (should be ~1): {y_scaled[train_mask_rows].std():.4f}")

    # -------------------------------------------
    # 11) Rebuild scaled DataFrame for sequences
    # -------------------------------------------
    print("[Step 11] Rebuilding scaled DataFrame for sequence building...")
    df_scaled = df_model[["area", "year"]].copy()
    df_scaled["target_scaled"] = y_scaled.reshape(-1)

    X_df = pd.DataFrame(X_scaled, columns=feature_cols, index=df_model.index)
    df_final = pd.concat([df_scaled, X_df], axis=1)
    print(f"        df_final shape: {df_final.shape}")
    print("        Sample df_final head():")
    print(df_final.head())

    # -------------------------------------------
    # 12) Sequence building with split by TARGET YEAR
    # -------------------------------------------
    print("[Step 12] Building sequences and splitting by TARGET YEAR...")

    def build_sequences_by_split(
        df_seq: pd.DataFrame,
        feat_cols: list,
        target_col_name: str,
        seq_len_inner: int,
    ):
        X_train_list, y_train_list = [], []
        X_val_list, y_val_list     = [], []
        X_test_list, y_test_list   = [], []

        for _, g in df_seq.groupby("area"):
            g = g.sort_values("year")
            feats = g[feat_cols].values.astype(np.float32)
            targs = g[target_col_name].values.astype(np.float32)
            years = g["year"].values

            if len(g) < seq_len_inner:
                continue

            # Sliding window: [t-seq_len+1 ... t] -> target at t
            for i in range(len(g) - seq_len_inner + 1):
                X_seq = feats[i : i + seq_len_inner]
                y_t   = targs[i + seq_len_inner - 1]
                y_yr  = years[i + seq_len_inner - 1]

                if y_yr < 2014:
                    X_train_list.append(X_seq)
                    y_train_list.append(y_t)
                elif 2014 <= y_yr <= 2018:
                    X_val_list.append(X_seq)
                    y_val_list.append(y_t)
                else:  # y_yr >= 2019
                    X_test_list.append(X_seq)
                    y_test_list.append(y_t)

        def to_array(x_list, y_list):
            if len(x_list) == 0:
                return (
                    np.empty((0, seq_len_inner, len(feat_cols)), dtype=np.float32),
                    np.empty((0,), dtype=np.float32),
                )
            X_arr = np.stack(x_list).astype(np.float32)
            y_arr = np.array(y_list, dtype=np.float32)
            return X_arr, y_arr

        X_tr, y_tr = to_array(X_train_list, y_train_list)
        X_v, y_v   = to_array(X_val_list, y_val_list)
        X_te, y_te = to_array(X_test_list, y_test_list)

        return X_tr, y_tr, X_v, y_v, X_te, y_te

    X_train, y_train, X_val, y_val, X_test, y_test = build_sequences_by_split(
        df_final,
        feature_cols,
        "target_scaled",
        seq_len,
    )

    print(f"        Sequence shapes:")
    print(f"           Train: {X_train.shape}, targets: {y_train.shape}")
    print(f"           Val:   {y_val.shape}, targets: {y_val.shape}")
    print(f"           Test:  {X_test.shape}, targets: {y_test.shape}")

    # -------------------------------------------
    # 13) Build PyTorch DataLoaders
    # -------------------------------------------
    print("[Step 13] Building PyTorch DataLoaders...")

    def make_loader(X, y, batch_size_inner: int, shuffle: bool):
        X_tensor = torch.from_numpy(X)  # (N, seq_len, num_features)
        y_tensor = torch.from_numpy(y).unsqueeze(-1)  # (N, 1)
        dataset = TensorDataset(X_tensor, y_tensor)
        return DataLoader(dataset, batch_size=batch_size_inner, shuffle=shuffle)

    train_loader = make_loader(X_train, y_train, batch_size, shuffle=True)
    val_loader   = make_loader(X_val,   y_val,   batch_size, shuffle=False)
    test_loader  = make_loader(X_test,  y_test,  batch_size, shuffle=False)

    print("========== PREPROCESSING DONE ==========")
    print(f"Train sequences: {len(train_loader.dataset)}")
    print(f"Val sequences:   {len(val_loader.dataset)}")
    print(f"Test sequences:  {len(test_loader.dataset)}")
    print("========================================")

    return train_loader, val_loader, test_loader, feature_cols, scaler_y


In [ ]:
def compute_baseline_rmse(df, target_col):
    """
    Compute baseline model: y(t) ≈ y(t-1)
    Using raw (unscaled) yearly panel data.
    """
    df_sorted = df.sort_values(["area", "year"]).copy()
    df_sorted["y_lag1"] = df_sorted.groupby("area")[target_col].shift(1)

    # Only test years
    test_mask = df_sorted["year"] >= 2019
    df_test = df_sorted[test_mask]

    y_true = df_test[target_col]
    y_pred = df_test["y_lag1"]

    # Drop rows where lag is not available (first year per country)
    mask = (~y_true.isna()) & (~y_pred.isna())
    y_true_clean = y_true[mask]
    y_pred_clean = y_pred[mask]

    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    r2 = r2_score(y_true_clean, y_pred_clean)

    return rmse, r2, y_true_clean.values, y_pred_clean.values


In [ ]:
# Build LSTM data loaders
train_loader, val_loader, test_loader, feature_cols, scaler_y = preprocess_and_build_loaders(
    df=df,
    chosen_crop=CHOSEN_CROP,
    target_col=TARGET_COL,
    seq_len=SEQ_LEN,
    batch_size=BATCH_SIZE,
    clip_quant=CLIP_QUANT
)

# ---- Compute baseline BEFORE training ----
rmse_baseline, r2_baseline, y_test_clean, y_pred_clean = compute_baseline_rmse(df, TARGET_COL)

print(f"\nBaseline model: RMSE={rmse_baseline:.2f}, R²={r2_baseline:.4f}")


In [ ]:
class TemporalFusionTransformer(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_heads=2, dropout=0.2):
        super().__init__()

        self.input_proj = nn.Linear(input_size, hidden_size)

        self.encoder_lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            dropout=0.0,  # only used if num_layers > 1
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, hidden_size),
        )

        # Optional decoder LSTM (you can remove this to simplify further)
        self.decoder_lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            dropout=0.0,
        )

        self.output_proj = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )

        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)

    def forward(self, x):
        """
        x: (batch, seq_len, input_size)
        returns: (batch, 1) – prediction in SCALED space
        """
        # Project inputs
        x = self.input_proj(x)                 # (B, T, H)

        # Encoder LSTM
        enc_out, (h, c) = self.encoder_lstm(x) # (B, T, H)

        # Self-attention
        attn_out, _ = self.attention(enc_out, enc_out, enc_out)  # (B, T, H)

        # Residual + norm
        x = self.norm1(enc_out + attn_out)

        # FFN + residual + norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)

        # Decoder LSTM (optional)
        dec_out, _ = self.decoder_lstm(x, (h, c))  # (B, T, H)

        # Take last time step
        last_step = dec_out[:, -1, :]  # (B, H)

        # Project to scalar
        out = self.output_proj(last_step)  # (B, 1)
        return out


In [ ]:
# ============================================================
# 3. TRAINING LOOP (WITH EARLY STOPPING, RMSE IN ORIGINAL UNITS)
# ============================================================
import copy


def rmse_unscaled(y_true_scaled, y_pred_scaled, scaler_y):
    """
    y_true_scaled, y_pred_scaled: numpy arrays in scaled space.
    Returns RMSE in ORIGINAL units.
    """
    y_true_scaled = y_true_scaled.reshape(-1, 1)
    y_pred_scaled = y_pred_scaled.reshape(-1, 1)

    y_true = scaler_y.inverse_transform(y_true_scaled).reshape(-1)
    y_pred = scaler_y.inverse_transform(y_pred_scaled).reshape(-1)

    return np.sqrt(mean_squared_error(y_true, y_pred))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

input_dim = len(feature_cols)
model = TemporalFusionTransformer(
    input_size=input_dim,
    hidden_size=32,
    num_heads=2,
    dropout=0.2,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

criterion = nn.MSELoss()

num_epochs = 100
history = {"train_rmse": [], "val_rmse": []}

best_val_rmse = float("inf")
best_state = None
patience = 20
no_improve = 0

for epoch in range(num_epochs):
    # ---------- TRAIN ----------
    model.train()
    train_y_true_scaled = []
    train_y_pred_scaled = []

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device).float()  # (B, T, F)
        y_batch = y_batch.to(device).float()  # (B, 1) – scaled

        optimizer.zero_grad()
        outputs = model(x_batch)              # (B, 1) – scaled

        loss = criterion(outputs, y_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        train_y_true_scaled.append(y_batch.detach().cpu().numpy())
        train_y_pred_scaled.append(outputs.detach().cpu().numpy())

    train_y_true_scaled = np.concatenate(train_y_true_scaled).reshape(-1)
    train_y_pred_scaled = np.concatenate(train_y_pred_scaled).reshape(-1)
    train_rmse = rmse_unscaled(train_y_true_scaled, train_y_pred_scaled, scaler_y)

    # ---------- VALIDATION ----------
    model.eval()
    val_y_true_scaled = []
    val_y_pred_scaled = []
    val_losses = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device).float()
            y_batch = y_batch.to(device).float()

            outputs = model(x_batch)

            loss = criterion(outputs, y_batch)
            val_losses.append(loss.item())

            val_y_true_scaled.append(y_batch.cpu().numpy())
            val_y_pred_scaled.append(outputs.cpu().numpy())

    if len(val_y_true_scaled) > 0:
        val_y_true_scaled = np.concatenate(val_y_true_scaled).reshape(-1)
        val_y_pred_scaled = np.concatenate(val_y_pred_scaled).reshape(-1)

        val_rmse = rmse_unscaled(val_y_true_scaled, val_y_pred_scaled, scaler_y)
        val_loss = np.mean(val_losses)
    else:
        val_rmse = np.nan
        val_loss = np.nan

    history["train_rmse"].append(train_rmse)
    history["val_rmse"].append(val_rmse)

    if not np.isnan(val_loss):
        scheduler.step(val_loss)

    # Early stopping on val_rmse
    if val_rmse < best_val_rmse - 1e-3:
        best_val_rmse = val_rmse
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    if epoch % 10 == 0 or epoch == num_epochs - 1:
        print(
            f"Epoch {epoch}/{num_epochs-1} | "
            f"Train RMSE: {train_rmse:.2f} | "
            f"Val RMSE: {val_rmse:.2f}"
        )

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch} (best val RMSE={best_val_rmse:.2f})")
        break

# Restore best model
if best_state is not None:
    model.load_state_dict(best_state)
    print("Loaded best model with Val RMSE:", best_val_rmse)

# ============================================================
# 5. PLOT TRAIN vs VAL RMSE
# ============================================================

epochs = range(1, len(history["train_rmse"]) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs, history["train_rmse"], label="Train RMSE", linewidth=2)
plt.plot(epochs, history["val_rmse"],   label="Val RMSE", linewidth=2)
plt.title("Transformer – Train vs Validation RMSE", fontsize=14)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("RMSE (original units)", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# ============================================================
# 4. EVALUATE ON TEST SET
# ============================================================

model.eval()
test_y_true_scaled = []
test_y_pred_scaled = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device).float()
        y_batch = y_batch.to(device).float()

        outputs = model(x_batch)

        test_y_true_scaled.append(y_batch.cpu().numpy())
        test_y_pred_scaled.append(outputs.cpu().numpy())

if len(test_y_true_scaled) > 0:
    test_y_true_scaled = np.concatenate(test_y_true_scaled).reshape(-1)
    test_y_pred_scaled = np.concatenate(test_y_pred_scaled).reshape(-1)
    test_rmse = rmse_unscaled(test_y_true_scaled, test_y_pred_scaled, scaler_y)
    print(f"Test RMSE (original units): {test_rmse:.2f}")
else:
    print("No test samples available.")

